# pytempo: a guided tour

`pytempo` reads Romanian official statistics from the INS TEMPO Online API:
it finds indicators, reads their metadata, and pulls the data into a pandas
DataFrame.

**This notebook runs live against the INS server.** Every cell makes real
requests, so it needs a network connection, and a few cells take a few
seconds. The examples were chosen to stay small and polite: no cell here
makes more than a couple of requests.

Install it straight from GitHub:

    pip install git+https://github.com/CIDS-UBB/pytempo.git


In [1]:
import pytempo as t

## 1. Discovery

Start with a panorama: how big the catalogue is and where to begin.


In [2]:
t.overview()

pytempo: 1916 TEMPO indicators, in 8 top level domains.
Start with find('salariati') or domains(). t.help() has the full guide.


`find` is the plain keyword search. It looks in the indicator name and in
its code, ignores diacritics and case, and requires every word you give it to
match. It returns **all** matches, not a truncated page, so slice the result
if you only want a few.


In [3]:
rezultate = t.find("salariati")
print(len(rezultate), "indicators")
rezultate[:5]

104 indicators


[Matrix('AMG1103', 'AMIGO - Populatia ocupata si salariatii dupa programul de lucru, pe grupe de varsta si sexe'),
 Matrix('AMG1104', 'AMIGO - Populatia ocupata si salariatii dupa programul de lucru, pe grupe de varsta si medii de rezidenta'),
 Matrix('AMG1105', 'AMIGO - Populatia ocupata si salariatii dupa programul de lucru, pe activitati si sexe'),
 Matrix('AMG1106', 'AMIGO - Populatia ocupata si salariatii dupa programul de lucru, pe activitati si medii de rezidenta'),
 Matrix('AMG115K', 'AMIGO - Salariati cu regim de lucru temporar dupa durata obisnuita a saptamanii de lucru si sexe')]

`search` is the other tool: discovery with filters. The keyword is optional,
and the filters combine with each other.

* `domeniu` matches a substring of the statistical domain name, so `economic`
  finds `B. STATISTICA ECONOMICA` without you knowing the exact wording.
* `periodicitate` matches a substring of how often the indicator is published.
* `level` keeps only indicators that reach that territorial level.
* `caen=True` keeps only those with a CAEN activity classification.

The difference in one line: **`find` searches names, `search` filters on
metadata.**


In [4]:
economici = t.search(domeniu="economic", periodicitate="anuala", level="judet")
print(len(economici), "indicators")
economici[:5]

111 indicators


[Matrix('AGR101A', 'Suprafata fondului funciar dupa modul de folosinta, pe forme de proprietate, macroregiuni, regiuni de dezvoltare si judete'),
 Matrix('AGR101B', 'Suprafata fondului funciar dupa modul de folosinta, pe judete si localitati'),
 Matrix('AGR102A', 'Suprafata terenurilor amenajate cu lucrari de irigatii si suprafata agricola irigata, pe categorii de folosinta a terenurilor, macroregiuni, regiuni de dezvoltare si judete'),
 Matrix('AGR102B', 'Suprafata terenurilor amenajate cu lucrari de desecare, pe categorii de folosinta a terenurilor, macroregiuni, regiuni de dezvoltare si judete'),
 Matrix('AGR102C', 'Suprafata terenurilor amenajate cu lucrari de ameliorare si combaterea eroziunii solului, pe categorii de folosinta a terenurilor, macroregiuni, regiuni de dezvoltare si judete')]

`t.filters()` prints what you can filter on, with the real values read from
the catalogue rather than from a hardcoded list.


In [5]:
t.filters()

Filters for t.search(). They combine with each other and with the search words.
  level        : ['national', 'macroregiune', 'regiune', 'judet', 'localitate', 'necunoscut']
  caen         : True only those with a CAEN dimension, False only those without
  domeniu      : substring of the domain name, diacritics ignored
                 A. STATISTICA SOCIALA
                 B. STATISTICA ECONOMICA
                 C. FINANTE
                 D. JUSTITIE
                 E. MEDIU INCONJURATOR
                 F. UTILITATI PUBLICE SI ADMINISTRAREA TERITORIULUI
                 G. DEZVOLTARE DURABILA - Orizont 2020
                 H. DEZVOLTARE DURABILA - Tinte 2030
  periodicitate: substring of the periodicity, diacritics ignored
                 ['5 - 6 ani', 'Anuala', 'Cincinal', 'La 2 ani', 'La fiecare 3 ani sau mai mult', 'La trei ani', 'Lunara', 'Perioada neregulata', 'Recomandabil la 10 ani', 'Trimestriala']

The metadata filters rest on the local index. If it is missing,
search a

## 2. Understanding one indicator

Before pulling data, read what the indicator actually is. We use FOM104D,
the average number of employees by county and locality.


In [6]:
m = t.matrix("FOM104D")
m

Matrix('FOM104D', 'Numarul mediu al salariatilor pe judete si localitati')

`what()` is the short version: the first sentence of the definition, the
unit, how often it is published, when it was last updated, and a warning if
the observations mention specific years, which usually signals a break in
the series.


In [7]:
m.what()

FOM104D  Numarul mediu al salariatilor pe judete si localitati
  Numarul mediu al salariatilor cuprinde persoanele angajate cu contract de munca/raport de serviciu pe durata determinata sau nedeterminata (inclusiv lucratorii sezonieri, managerul sau administratorul), al caror contract de munca/raport de serviciu nu a fost suspendat in perioada de referinta.
  unit        : Numar persoane
  periodicity : Anuala
  updated     : 20-11-2025
                read them with .describe()


`where()` shows where the indicator sits in the domain tree, and what it
covers: how many territorial units at each level, whether localities carry a
SIRUTA code, and the span of years.

A note on **territorial levels**. `national`, `macroregiune`, `regiune`,
`judet`, `localitate` are how pytempo interprets the option names, not a
concept INS exposes directly. A territorial dimension usually mixes all of
them in one column, and pytempo works out which is which so you can ask for
one. Names that do not fit the administrative nomenclator, such as monitoring
stations, are labelled `necunoscut` rather than being forced into a level.


In [8]:
m.where()

domain   : A. STATISTICA SOCIALA > FORTA DE MUNCA > SALARIATI
territory: Judete (43 options)
    national        1
    judet           42
territory: Localitati (3183 options)
    localitate      3183
SIRUTA   : yes
time     : Ani, 35 periods, 1990 to 2024


`how()` generates the download manual for this specific indicator: the
commands that make sense for it, the strategy that will be used, and how many
requests to expect.


In [9]:
m.how()

How to download FOM104D:
  m = t.matrix('FOM104D')
  df = m.get()          level localitate, tidied
  (the level filter does not apply here: county and locality are
   separate dimensions, and get() brings both anyway)
  m.get(raw=True)       exactly what INS returns, no extras

  strategy: by_county, roughly 43 requests
  downloaded in several requests and concatenated


`describe()` prints the full record exactly as INS wrote it: the complete
definition, the methodology, the sources and the observations. It is long on
purpose. The observations are where series breaks and warnings about
incomplete years live, so read them before trusting a series.


In [10]:
m.describe()

FOM104D  Numarul mediu al salariatilor pe judete si localitati
domain      : A. STATISTICA SOCIALA > FORTA DE MUNCA > SALARIATI
levels      : national, judet, localitate
periodicity : Anuala
updated     : 20-11-2025

DEFINITION
Numarul mediu al salariatilor cuprinde persoanele angajate cu contract de munca/raport de serviciu pe durata determinata sau nedeterminata (inclusiv lucratorii sezonieri, managerul sau administratorul), al caror contract de munca/raport de serviciu nu a fost suspendat in perioada de referinta.
Numarul mediu al salariatilor se calculeaza ca medie aritmetica simpla rezultata din suma efectivelor zilnice de salariati (exclusiv cei al caror contract de munca/raport de serviciu a fost suspendat), din perioada de referinta, inclusiv din zilele de repaus saptamanal, sarbatori legale si alte zile nelucratoare, impartita la numarul total al zilelor calendaristice.
In efectivul zilnic al salariatilor luat in calculul numarului mediu se cuprind urmatoarele categorii:
- sal

`options()` with no argument lists the dimensions, each with the role
pytempo assigned to it and how many values it has.


In [11]:
m.options()

[0] Judete (teritoriu, 43 options)
[1] Localitati (teritoriu, 3183 options)
[2] Ani (timp, 35 options)
[3] UM: Numar persoane (um, 1 options)

With an argument it lists the values of one dimension. You can name it by
label, by role, by index, or by level.


In [12]:
m.options("teritoriu", limit=8)

TOTAL, 1017 MUNICIPIUL ALBA IULIA, 1213 MUNICIPIUL AIUD, 1348 MUNICIPIUL BLAJ, 1874 MUNICIPIUL SEBES, 1151 ORAS ABRUD, 2915 ORAS BAIA DE ARIES, 1455 ORAS CAMPENI

## 3. A simple extraction

FOM101A, labour resources by county, fits in a single request, so it is a
good first pull.

`get()` with no arguments does three things by default: it picks the finest
territorial level the indicator actually reaches, it applies the tidy
standardization, and it prints one line saying what it decided.


In [13]:
df = t.matrix("FOM101A").get()
df.shape

FOM101A: level judet (the finest), single, 1 request
  for every level, including national, macroregiune and regiune, use get(level=None)


(4392, 7)

The result is in **long format**: one row per combination, one text column
per dimension, a numeric `Valoare` column, and then the derived columns that
tidy added.


In [14]:
df.head()

,Sexe,"Macroregiuni, regiuni de dezvoltare si judete",Ani,UM: Mii persoane,Valoare,"Macroregiuni, regiuni de dezvoltare si judete_nivel",Ani_an
0,Total,Arges,Anul 1990,Mii persoane,394.8,judet,1990
1,Total,Arges,Anul 1991,Mii persoane,394.7,judet,1991
2,Total,Arges,Anul 1992,Mii persoane,399.5,judet,1992
3,Total,Arges,Anul 1993,Mii persoane,392.9,judet,1993
4,Total,Arges,Anul 1994,Mii persoane,392.2,judet,1994


The derived columns are typed, not text: `Int64` for the SIRUTA code and the
year, nullable strings for the level, the type and the clean name.


In [15]:
df.dtypes

Sexe                                                       str
Macroregiuni, regiuni de dezvoltare si judete              str
Ani                                                        str
UM: Mii persoane                                           str
Valoare                                                float64
Macroregiuni, regiuni de dezvoltare si judete_nivel     string
Ani_an                                                   Int64
dtype: object

Asking for a level changes what comes back. Counties and regions are
different slices of the same indicator, so the row counts differ.


In [16]:
judete = t.matrix("FOM101A").get(level="judet", progress=False)
regiuni = t.matrix("FOM101A").get(level="regiune", progress=False)
print("judet  :", judete.shape)
print("regiune:", regiuni.shape)

judet  : (4392, 7)
regiune: (840, 7)


`raw=True` gives exactly what INS returned, with no derived columns. Use raw
when you want to see the source untouched or you are writing your own
processing; use tidy, the default, when you want to work with the data.


In [17]:
brut = t.matrix("FOM101A").get(raw=True, progress=False)
print("raw :", brut.shape)
print("tidy:", df.shape)
brut.head(3)

raw : (4392, 5)
tidy: (4392, 7)


,Sexe,"Macroregiuni, regiuni de dezvoltare si judete",Ani,UM: Mii persoane,Valoare
0,Total,Arges,Anul 1990,Mii persoane,394.8
1,Total,Arges,Anul 1991,Mii persoane,394.7
2,Total,Arges,Anul 1992,Mii persoane,399.5


## 4. Standardization and SIRUTA

SIRUTA is the official code of a Romanian administrative unit. INS puts it
inside the locality name, as a numeric prefix, so the raw label looks like
`1017 MUNICIPIUL ALBA IULIA`.

pytempo splits that into separate columns and **keeps the original label
untouched**. SIRUTA is preserved as a key, never dropped, because it is what
lets you join this data with other administrative sources: population
registers, budgets, geographies.

We use SAN103B here, children enrolled in nurseries by county and locality,
because it reaches locality level and still fits in a single request.


In [18]:
crese = t.matrix("SAN103B").get()
localitati = crese[crese["Localitati_nivel"] == "localitate"]
localitati[["Localitati", "Localitati_siruta", "Localitati_tip",
            "Localitati_nume", "Valoare"]].head(6)

SAN103B: all levels, single, 1 request


,Localitati,Localitati_siruta,Localitati_tip,Localitati_nume,Valoare
2,1017 MUNICIPIUL ALBA IULIA,1017,municipiu,ALBA IULIA,50
3,1213 MUNICIPIUL AIUD,1213,municipiu,AIUD,41
5,9262 MUNICIPIUL ARAD,9262,municipiu,ARAD,159
6,9459 ORAS CHISINEU-CRIS,9459,oras,CHISINEU-CRIS,14
7,9538 ORAS INEU,9538,oras,INEU,18
8,9574 ORAS LIPOVA,9574,oras,LIPOVA,50


Notice what happened: the code, the type of settlement and the clean name are
now three separate, typed columns, while the `Localitati` column still holds
the original text.

One more thing to know: **the data is sparse.** Combinations with no data are
absent as whole rows, they do not arrive as `NaN`. That reflects real
administrative history rather than a gap in the library: Ilfov and Municipiul
Bucuresti do not exist as separate units before 1996, so those rows simply do
not exist. Do not check a download by comparing the row count against the
product of the dimensions.


In [19]:
print("rows returned :", len(crese))
print("empty values  :", int(crese["Valoare"].isna().sum()))

rows returned : 180
empty values  : 0


## 5. A larger download, with automatic splitting

**This cell takes longer than the others.** It makes more than one request.

A single POST to INS is capped at a cell budget. When an indicator is larger
than that, pytempo splits the work automatically and concatenates the pieces:
county by county for indicators that reach locality level, otherwise on the
largest dimension. You do not have to plan anything, but you should know it is
happening, which is why `get()` prints the decision line.

FOM106E is a good demonstration: it splits on the CAEN dimension into a
couple of requests. Indicators that would take hundreds of requests are not
used in this tutorial, and `get()` asks for confirmation before starting one.


In [20]:
big = t.matrix("FOM106E").get()
big.shape

FOM106E: level judet (the finest), split:CAEN Rev.2  (activitati ale economiei nationale), 2 requests
  for every level, including national, macroregiune and regiune, use get(level=None)


  1/2: +84354 rows (total 84354)


  2/2: +45256 rows (total 129610)


(129610, 8)

## 6. A small analysis

The data comes back ready to use. Nothing below is pytempo specific: from
here on it is ordinary pandas.

We take the tidy FOM101A frame, keep one county, and read the series by year.


In [21]:
terr = "Macroregiuni, regiuni de dezvoltare si judete"
cluj = df[(df[terr] == "Cluj") & (df["Sexe"] == "Total")]
cluj = cluj.sort_values("Ani_an")
cluj[[terr, "Ani_an", "Valoare"]].tail(10)

,"Macroregiuni, regiuni de dezvoltare si judete",Ani_an,Valoare
305,Cluj,2015,464.4
306,Cluj,2016,471.0
307,Cluj,2017,468.8
308,Cluj,2018,464.9
309,Cluj,2019,465.5
310,Cluj,2020,467.8
311,Cluj,2021,472.0
312,Cluj,2022,437.1
313,Cluj,2023,443.7
314,Cluj,2024,447.2


No plots here, on purpose: this notebook adds no dependencies beyond what
pytempo already needs. Plotting, mapping and modelling are ordinary work on
an ordinary DataFrame.


In [22]:
cluj["Valoare"].describe()

count     35.000000
mean     456.131429
std       10.760597
min      434.300000
25%      447.450000
50%      459.500000
75%      464.650000
max      472.000000
Name: Valoare, dtype: float64

## Where to go next

* `t.help()` prints the full navigation guide.
* `m.help()` does the same for one indicator.
* `m.how()` gives you the download commands for that specific indicator.
* The README covers levels, roles, the shape of the data and the internal
  schema registry.

One closing thought. pytempo does one job: getting the data out of TEMPO,
correctly and reproducibly. Analysis, visualisation and mapping are not its
job, and it deliberately stays out of the way so you can use the usual tools.
